In [ ]:
import sys
import numpy
import scipy
import torch
import torchaudio

print("Python:", sys.version)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)
print("PyTorch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("GPU available:", torch.cuda.is_available())

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
NumPy: 2.4.6
SciPy: 1.16.3
PyTorch: 2.14.1+cu130
Torchaudio: 2.11.0+cu130
GPU available: True


In [ ]:
# Cell 1 — INSTALL
# After completion: Runtime → Restart session

import os
import sys
import subprocess

print("Python:", sys.version, flush=True)

OPENVOICE_REPO = "/content/OpenVoice"
OPENVOICE_COMMIT = "74a1d147b17a8c3092dd5430504bd83ef6c7eb23"

SEED_REPO = "/content/seed-vc"
SEED_COMMIT = "51383efd921027683c89e5348211d93ff12ac2a8"

repositories = [
    (
        OPENVOICE_REPO,
        "https://github.com/myshell-ai/OpenVoice.git",
        OPENVOICE_COMMIT,
    ),
    (
        SEED_REPO,
        "https://github.com/Plachtaa/seed-vc.git",
        SEED_COMMIT,
    ),
]

for repo, url, commit in repositories:
    print(f"\nPreparing {repo}...", flush=True)

    if not os.path.isdir(repo):
        subprocess.run(
            ["git", "clone", "--progress", url, repo],
            check=True,
        )

    subprocess.run(
        ["git", "checkout", commit],
        cwd=repo,
        check=True,
    )

    print(f"Repository ready: {repo}", flush=True)

pkgs = [
    # OpenVoice
    "wavmark==0.0.3",
    "eng_to_ipa==0.0.2",
    "inflect==7.0.0",
    "unidecode==1.3.7",
    "pypinyin==0.50.0",
    "cn2an==0.5.22",
    "jieba==0.42.1",
    "langid==1.1.6",

    # Seed-VC
    "numpy==1.26.4",
    "scipy==1.13.1",
    "librosa==0.10.2",
    "transformers==4.46.3",
    "huggingface-hub>=0.28.1",
    "munch==4.0.0",
    "einops==0.8.0",
    "descript-audio-codec==1.0.0",
    "resemblyzer",
    "soundfile==0.12.1",
    "modelscope==1.18.1",
    "funasr==1.1.5",
    "hydra-core==1.3.2",
    "python-dotenv",
    "pyyaml",

    # Scoring
    "speechbrain==1.0.3",
    "jiwer==3.0.3",
    "accelerate",
]

print("\nInstalling packages...", flush=True)

process = subprocess.Popen(
    [
        sys.executable,
        "-u",
        "-m",
        "pip",
        "install",
        "--progress-bar",
        "off",
    ] + pkgs,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="", flush=True)

return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Package installation failed: {return_code}"
    )

print(
    "\n✅ Installed. NOW: Runtime → Restart session, "
    "then run Cell 2.",
    flush=True,
)

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

Preparing /content/OpenVoice...
Repository ready: /content/OpenVoice

Preparing /content/seed-vc...
Repository ready: /content/seed-vc

Installing packages...
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requireme

In [ ]:
# Cell 2 — PATHS AND SETTINGS
import os, sys, glob, json, time, random, shutil, tempfile, re, string, unicodedata, subprocess
os.environ["USE_TF"] = "0"; os.environ["TRANSFORMERS_NO_TF"] = "1"; os.environ["USE_FLAX"] = "0"
from google.colab import drive
drive.mount("/content/drive")
import numpy as np, pandas as pd, torch

BASE_DIR = "/content/drive/MyDrive/IJEEE_VC_data"
OUT = os.path.join(BASE_DIR, "revision_runs_final"); os.makedirs(OUT, exist_ok=True)

TUNING_FILES = [os.path.join(BASE_DIR, "selected_hybrid_data", f"joe_source_{i:02d}.wav") for i in (1, 2, 3)]
TUNING_TEXT = ["The monkey was planning to throw the net like the fishermen did.",
               "Gaze out to sea, picnic on benches and watch fishermen hook herring.",
               "We all have our own methods of dealing with money, but here are a couple of saving tricks that will grow your money."]
TARGET_REF = os.path.join(BASE_DIR, "selected_hybrid_data", "kathleen_reference.wav")
# EDIT if the two new conditioning references are stored elsewhere on Drive:
EXTRA_REFS = [os.path.join(BASE_DIR, "selected_hybrid_data", f"kathleen_conditioning_ref_{i:02d}.wav") for i in (2, 3)]
EVAL_CLIPS = sorted(glob.glob(os.path.join(BASE_DIR, "kathleen_independent_refs", "kathleen_eval_*.wav")))
JOE_ARCHIVE = os.path.join(BASE_DIR, "joe.tar.gz")
HOLDOUT10_META = os.path.join(BASE_DIR, "final_holdout_data", "holdout_metadata.csv")
HOLDOUT40_DIR = os.path.join(BASE_DIR, "final_holdout_data_40"); os.makedirs(HOLDOUT40_DIR, exist_ok=True)
HOLDOUT40_META = os.path.join(HOLDOUT40_DIR, "holdout40_metadata.csv")
EXISTING = os.path.join(BASE_DIR, "final_hybrid_comparison")
OV_CONVERTER = os.path.join(BASE_DIR, "models", "OpenVoiceV2", "converter")
N_HOLDOUT = 40
TUNING_IDS = {"0000000160", "0000000229", "0000000068"}

for p in TUNING_FILES + [TARGET_REF, JOE_ARCHIVE, HOLDOUT10_META, os.path.join(OV_CONVERTER, "checkpoint.pth")]:
    assert os.path.isfile(p), f"❌ missing: {p}"
for p in EXTRA_REFS:
    print("extra reference", os.path.basename(p), "found" if os.path.isfile(p) else "❌ NOT FOUND — fix the EDIT line")
print("Independent Kathleen evaluation clips:", len(EVAL_CLIPS))
assert len(EVAL_CLIPS) >= 5, "❌ kathleen_eval_*.wav clips not found"
assert torch.cuda.is_available(), "❌ select a T4 GPU runtime"
print("GPU:", torch.cuda.get_device_name(0))

def set_all_seeds(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
def append_row(csv, row):
    pd.DataFrame([row]).to_csv(csv, mode="a", header=not os.path.exists(csv), index=False)
def done_keys(csv, cols):
    if not os.path.exists(csv): return set()
    return set(map(tuple, pd.read_csv(csv)[cols].astype(str).values))
print("✅ Paths ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
extra reference kathleen_conditioning_ref_02.wav found
extra reference kathleen_conditioning_ref_03.wav found
Independent Kathleen evaluation clips: 10
GPU: Tesla T4
✅ Paths ready


In [ ]:
# Cell 3 — EXTEND THE HOLDOUT SET TO 40 SENTENCES

from pathlib import Path
import tarfile

LOCAL_JOE = "/content/final_comparison_dataset/joe"
os.makedirs(LOCAL_JOE, exist_ok=True)

if not glob.glob(
    os.path.join(LOCAL_JOE, "**", "*.webm"),
    recursive=True,
):
    with tarfile.open(JOE_ARCHIVE, "r:gz") as archive:
        archive.extractall(LOCAL_JOE, filter="data")

files = []

for extension in [
    "*.webm", "*.wav", "*.flac", "*.mp3",
    "*.ogg", "*.opus", "*.m4a",
]:
    files.extend(
        glob.glob(
            os.path.join(LOCAL_JOE, "**", extension),
            recursive=True,
        )
    )

files = sorted(set(files))
print("Joe files:", len(files))

candidates = [
    path
    for path in files
    if Path(path).stem not in TUNING_IDS
]

random.Random(20260822).shuffle(candidates)


def dur(path):
    result = subprocess.run(
        [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            path,
        ],
        capture_output=True,
        text=True,
    )

    try:
        return float(result.stdout.strip())
    except ValueError:
        return None


def txt(path):
    transcript_path = Path(path).with_suffix(".txt")

    if not transcript_path.is_file():
        return None

    transcript = " ".join(
        transcript_path.read_text(
            encoding="utf-8",
            errors="ignore",
        ).split()
    )

    return transcript or None


records = []

for path in candidates:
    if len(records) == N_HOLDOUT:
        break

    duration = dur(path)

    if duration is None or not (4.0 < duration < 7.5):
        continue

    transcript = txt(path)

    if transcript is None:
        continue

    sample_number = len(records) + 1

    output_path = os.path.join(
        HOLDOUT40_DIR,
        f"joe_holdout_{sample_number:02d}.wav",
    )

    if not os.path.isfile(output_path):
        subprocess.run(
            [
                "ffmpeg",
                "-y",
                "-v", "error",
                "-i", path,
                "-ac", "1",
                "-ar", "22050",
                "-c:a", "pcm_s16le",
                output_path,
            ],
            check=True,
        )

    records.append(
        {
            "sample": sample_number,
            "original_id": Path(path).stem,
            "duration_seconds": duration,
            "transcript": transcript,
            "prepared_wav": output_path,
        }
    )

    print(
        f"Prepared {sample_number}/{N_HOLDOUT}: "
        f"{Path(path).stem}",
        flush=True,
    )

assert len(records) == N_HOLDOUT, (
    f"❌ Expected {N_HOLDOUT} sentences, found {len(records)}"
)

H40 = pd.DataFrame(records)

# Preserve leading zeros in recording IDs.
old10 = pd.read_csv(
    HOLDOUT10_META,
    dtype={"original_id": str},
)

same = (
    H40["original_id"].astype(str).head(10).tolist()
    == old10["original_id"].astype(str).tolist()
)

print("First 10 identical to the paper's holdout:", same)

assert same, (
    "❌ The extended selection does not start with "
    "the original 10 — stop and report"
)

H40.to_csv(HOLDOUT40_META, index=False)

print(
    H40[
        ["sample", "original_id", "duration_seconds"]
    ].tail(5)
)

print("✅ 40 holdout sentences ready")

Joe files: 1150
Prepared 1/40: 4000000077
Prepared 2/40: 3000000204
Prepared 3/40: 0000000107
Prepared 4/40: 4000000151
Prepared 5/40: 0300000043
Prepared 6/40: 0000000002
Prepared 7/40: 4000000149
Prepared 8/40: 0000000280
Prepared 9/40: 4000000236
Prepared 10/40: 0000000031
Prepared 11/40: 4000000070
Prepared 12/40: 4000000071
Prepared 13/40: 4000000107
Prepared 14/40: 0000000167
Prepared 15/40: 4000000017
Prepared 16/40: 0000000155
Prepared 17/40: 0300000009
Prepared 18/40: 4000000279
Prepared 19/40: 0000000269
Prepared 20/40: 4000000082
Prepared 21/40: 4000000253
Prepared 22/40: 0300000029
Prepared 23/40: 0300000044
Prepared 24/40: 0000000293
Prepared 25/40: 0000000250
Prepared 26/40: 4000000141
Prepared 27/40: 4000000009
Prepared 28/40: 0100000065
Prepared 29/40: 4000000058
Prepared 30/40: 0000000190
Prepared 31/40: 0000000052
Prepared 32/40: 0000000283
Prepared 33/40: 4000000219
Prepared 34/40: 0000000125
Prepared 35/40: 3000000002
Prepared 36/40: 4000000258
Prepared 37/40: 00000

In [ ]:
import os

for path in [
    "/content/OpenVoice",
    "/content/OpenVoice/openvoice",
    "/content/OpenVoice/openvoice/api.py",
    "/content/seed-vc",
]:
    print(path, "→", os.path.exists(path))

/content/OpenVoice → True
/content/OpenVoice/openvoice → True
/content/OpenVoice/openvoice/api.py → True
/content/seed-vc → True


In [ ]:
import sys
import importlib
import importlib.util

repo = "/content/OpenVoice"

if repo not in sys.path:
    sys.path.insert(0, repo)

importlib.invalidate_caches()

spec = importlib.util.find_spec("openvoice")
print("OpenVoice location:", spec)

from openvoice.api import ToneColorConverter

print("✅ OpenVoice imported successfully")

OpenVoice location: ModuleSpec(name='openvoice', loader=<_frozen_importlib_external.SourceFileLoader object at 0x7e5fa52ab1d0>, origin='/content/OpenVoice/openvoice/__init__.py', submodule_search_locations=['/content/OpenVoice/openvoice'])


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/jieba/_compat.py:18: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


✅ OpenVoice imported successfully


/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
# Cell 4 — LOAD MODELS, WARM-UP, CONVERSION FUNCTIONS (from Untitled15)
from types import SimpleNamespace
sys.path.insert(0, "/content/OpenVoice")
from openvoice.api import ToneColorConverter
tone_color_converter = ToneColorConverter(os.path.join(OV_CONVERTER, "config.json"), device="cuda:0")
tone_color_converter.load_ckpt(os.path.join(OV_CONVERTER, "checkpoint.pth"))
sys.path.insert(0, "/content/seed-vc"); os.chdir("/content/seed-vc")
import inference as seed_inference
def seed_args(src, ref, out, steps):
    return SimpleNamespace(source=src, target=ref, output=out, diffusion_steps=steps, length_adjust=1.0,
                           inference_cfg_rate=0.7, f0_condition=False, auto_f0_adjust=False, semi_tone_shift=0,
                           checkpoint=None, config=None, fp16=True)
seed_model_bundle = seed_inference.load_models(seed_args(None, None, None, 10))
seed_inference.load_models = lambda args: seed_model_bundle       # never reload inside main()

SE_CACHE = {}
def get_se(path):                     # cached, outside the timed span (as in the paper)
    if path not in SE_CACHE: SE_CACHE[path] = tone_color_converter.extract_se(path)
    return SE_CACHE[path]
def run_openvoice(src, ref, tau, out_path, seed):
    src_se, tgt_se = get_se(src), get_se(ref)
    set_all_seeds(seed); torch.cuda.synchronize(); t0 = time.perf_counter()
    tone_color_converter.convert(audio_src_path=src, src_se=src_se, tgt_se=tgt_se, output_path=out_path, tau=tau, message="@MyShell")
    torch.cuda.synchronize(); return time.perf_counter() - t0
os.makedirs("/content/seed_tmp", exist_ok=True)
def run_seedvc(src, ref, steps, out_path, seed):
    work = tempfile.mkdtemp(dir="/content/seed_tmp")
    set_all_seeds(seed); torch.cuda.synchronize(); t0 = time.perf_counter()
    seed_inference.main(seed_args(src, ref, work, steps))
    torch.cuda.synchronize(); el = time.perf_counter() - t0
    produced = sorted(glob.glob(os.path.join(work, "*.wav")), key=os.path.getmtime)
    assert produced, "❌ Seed-VC produced no file"
    shutil.move(produced[-1], out_path); shutil.rmtree(work, ignore_errors=True)
    return el
def run_cascade_e2e(src, ref, tau, steps, out_path, ov_seed, sv_seed):
    # one end-to-end timed run: both stages and the intermediate file inside one timed span
    inter = out_path.replace(".wav", "_e2e_ov.wav")
    torch.cuda.synchronize(); t0 = time.perf_counter()
    run_openvoice(src, ref, tau, inter, ov_seed); run_seedvc(inter, ref, steps, out_path, sv_seed)
    torch.cuda.synchronize(); return time.perf_counter() - t0

WARM = "/content/warmup"; os.makedirs(WARM, exist_ok=True)
for _ in range(2):
    run_openvoice(TUNING_FILES[0], TARGET_REF, 0.5, f"{WARM}/ov.wav", 2026)
    run_seedvc(TUNING_FILES[0], TARGET_REF, 10, f"{WARM}/sv.wav", 2028)
ENV = dict(gpu=torch.cuda.get_device_name(0), torch=torch.__version__, openvoice_commit="74a1d147b17a8c3092dd5430504bd83ef6c7eb23",
           seedvc_commit="51383efd921027683c89e5348211d93ff12ac2a8", seedvc_fp16=True, openvoice_precision="fp32",
           batch_size=1, warmups=2, timed_repeats=3, cuda_synchronize=True, openvoice_embeddings_cached=True)
json.dump(ENV, open(os.path.join(OUT, "environment.json"), "w"), indent=1)
print("✅ Models loaded and warmed up"); print(ENV)

/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:145: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


step59000_snr39.99_pesq4.35_BERP_none0.3(…):   0%|          | 0.00/10.0M [00:00<?, ?B/s]

Loaded checkpoint '/content/drive/MyDrive/IJEEE_VC_data/models/OpenVoiceV2/converter/checkpoint.pth'
missing/unexpected keys: [] []


DiT_seed_v2_uvit_whisper_small_wavenet_b(…):   0%|          | 0.00/440M [00:00<?, ?B/s]

(…)_mel_seed_uvit_whisper_small_wavenet.yml: 0.00B [00:00, ?B/s]

cfm loaded
length_regulator loaded


... (10 outputs left out)

It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
100%|██████████| 10/10 [00:00<00:00, 13.77it/s]


RTF: 0.35077800438956963


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
100%|██████████| 10/10 [00:00<00:00, 20.53it/s]


RTF: 0.21324246614805867
✅ Models loaded and warmed up
{'gpu': 'Tesla T4', 'torch': '2.14.1+cu130', 'openvoice_commit': '74a1d147b17a8c3092dd5430504bd83ef6c7eb23', 'seedvc_commit': '51383efd921027683c89e5348211d93ff12ac2a8', 'seedvc_fp16': True, 'openvoice_precision': 'fp32', 'batch_size': 1, 'warmups': 2, 'timed_repeats': 3, 'cuda_synchronize': True, 'openvoice_embeddings_cached': True}


In [ ]:
import numpy as np
import scipy
import scipy.signal
import scipy.linalg

print("NumPy:", np.__version__)
print("SciPy:", scipy.__version__)
print("✅ Imports passed")

NumPy: 1.26.4
SciPy: 1.13.1
✅ Imports passed


In [ ]:
# Cell 11 — SECOND-SESSION TIMING (timing only, no scoring)
H40 = pd.read_csv(HOLDOUT40_META)
csv = os.path.join(OUT, "E11_session_timing.csv"); session = time.strftime("%Y-%m-%d_%H%M")
d11 = os.path.join("/content", "E11_audio"); os.makedirs(d11, exist_ok=True)
for name, kind, tau, steps in [("OpenVoice V2 (tau 0.5)", "ov", 0.5, None), ("Seed-VC (10 steps)", "sv", None, 10),
                               ("Reference cascade (tau 0.3, 20)", "casc", 0.3, 20), ("Selected cascade (tau 0.5, 10)", "casc", 0.5, 10)]:
    for _, r in H40.head(10).iterrows():
        i = int(r["sample"]); out = os.path.join(d11, f"x_s{i}.wav"); inter = out.replace(".wav", "_ov.wav")
        t_ov = float(np.mean([run_openvoice(r.prepared_wav, TARGET_REF, tau, inter, 202700 + i) for _ in range(3)])) if kind != "sv" else 0.0
        t_sv = float(np.mean([run_seedvc(inter if kind == "casc" else r.prepared_wav, TARGET_REF, steps, out, 202900 + i) for _ in range(3)])) if kind != "ov" else 0.0
        e2e = run_cascade_e2e(r.prepared_wav, TARGET_REF, tau, steps, out, 202700 + i, 202900 + i) if kind == "casc" else np.nan
        append_row(csv, dict(session=session, gpu=torch.cuda.get_device_name(0), system=name, sample=i,
                             t_openvoice=t_ov, t_seedvc=t_sv, latency=t_ov + t_sv, latency_e2e=e2e))
print(pd.read_csv(csv).groupby(["session", "system"])[["latency", "latency_e2e"]].mean().round(3))

It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
100%|██████████| 10/10 [00:00<00:00, 20.18it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.2164536104624586


100%|██████████| 10/10 [00:00<00:00, 20.20it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.21742252909325968


100%|██████████| 10/10 [00:00<00:00, 20.20it/s]
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.


RTF: 0.21769165183269532


... (210 outputs left out)

100%|██████████| 10/10 [00:00<00:00, 16.37it/s]


RTF: 0.20631757797673345


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
100%|██████████| 10/10 [00:00<00:00, 16.20it/s]


RTF: 0.20912907784804705
                                                 latency  latency_e2e
session         system                                               
2026-11-15_0954 OpenVoice V2 (tau 0.5)             0.261          NaN
                Reference cascade (tau 0.3, 20)    2.066        2.070
                Seed-VC (10 steps)                 1.232          NaN
                Selected cascade (tau 0.5, 10)     1.479        1.486
2026-11-16_1327 OpenVoice V2 (tau 0.5)             0.242          NaN
                Reference cascade (tau 0.3, 20)    2.014        2.021
                Seed-VC (10 steps)                 1.154          NaN
                Selected cascade (tau 0.5, 10)     1.448        1.458


In [ ]:
timing = pd.read_csv(
    os.path.join(OUT, "E11_session_timing.csv")
)

print(
    timing.groupby(["session", "system"])
    .agg(rows=("sample", "size"), samples=("sample", "nunique"))
)

                                                 rows  samples
session         system                                        
2026-11-15_0954 OpenVoice V2 (tau 0.5)             10       10
                Reference cascade (tau 0.3, 20)    10       10
                Seed-VC (10 steps)                 10       10
                Selected cascade (tau 0.5, 10)     10       10
2026-11-16_1327 OpenVoice V2 (tau 0.5)             10       10
                Reference cascade (tau 0.3, 20)    10       10
                Seed-VC (10 steps)                 10       10
                Selected cascade (tau 0.5, 10)     10       10


In [ ]:

# Cell 13 — PACK RESULTS (CSVs + environment) AND DOWNLOAD
import zipfile
from google.colab import files
zp = "/content/IJEEE_revision_results_final.zip"
with zipfile.ZipFile(zp, "w") as z:
    for f in glob.glob(os.path.join(OUT, "*.csv")) + glob.glob(os.path.join(OUT, "*.json")) + [HOLDOUT40_META]:
        z.write(f, os.path.basename(f))
print("Files in zip:", len(zipfile.ZipFile(zp).namelist()))
files.download(zp)

Files in zip: 13


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
required = [
    "tone_color_converter",
    "seed_model_bundle",
    "run_openvoice",
    "run_seedvc",
    "run_cascade_e2e",
    "H40",
    "score",
    "speaker_model",
    "ASR",
]

for name in required:
    exists = name in globals()
    print(f"{'✅' if exists else '❌'} {name}")

import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

✅ tone_color_converter
✅ seed_model_bundle
✅ run_openvoice
✅ run_seedvc
✅ run_cascade_e2e
✅ H40
❌ score
❌ speaker_model
❌ ASR
GPU available: True
GPU: Tesla T4


In [ ]:
# Cell 5 — SCORING MODELS (ECAPA and Whisper-small exactly as in Untitled13; plus WavLM, Whisper-large-v3, UTMOS)
import librosa, jiwer, torchaudio, logging
import torch.nn.functional as F
if not hasattr(torchaudio, "list_audio_backends"):
    torchaudio.list_audio_backends = lambda: ["soundfile"]
from speechbrain.inference.speaker import SpeakerRecognition
from transformers import AutoProcessor, AutoModelForSpeechSeq2Seq, AutoFeatureExtractor, WavLMForXVector
logging.getLogger("speechbrain").setLevel(logging.WARNING)
DEV = "cuda"
speaker_model = SpeakerRecognition.from_hparams(source="speechbrain/spkrec-ecapa-voxceleb",
                                                savedir="/content/evaluation_models/ecapa_voxceleb", run_opts={"device": DEV})
ASR = {}
for name, mid, dtype in [("small", "openai/whisper-small", torch.float32), ("large", "openai/whisper-large-v3", torch.float16)]:
    proc = AutoProcessor.from_pretrained(mid)
    mdl = AutoModelForSpeechSeq2Seq.from_pretrained(mid, torch_dtype=dtype, use_safetensors=True).to(DEV).eval()
    mdl.config.forced_decoder_ids = None; mdl.generation_config.forced_decoder_ids = None
    ASR[name] = (proc, mdl, dtype)
wl_fe = AutoFeatureExtractor.from_pretrained("microsoft/wavlm-base-plus-sv")
wl = WavLMForXVector.from_pretrained("microsoft/wavlm-base-plus-sv").to(DEV).eval()
utmos = torch.hub.load("tarepan/SpeechMOS:v1.2.0", "utmos22_strong", trust_repo=True).to(DEV).eval()

def load_audio_16k(p): return np.asarray(librosa.load(p, sr=16000, mono=True)[0], dtype=np.float32)
_EMB = {}
@torch.inference_mode()
def emb(p, kind):
    if (p, kind) in _EMB: return _EMB[(p, kind)]
    w = load_audio_16k(p)
    if kind == "ecapa":   # identical to Untitled13 extract_speaker_embedding
        e = speaker_model.encode_batch(torch.from_numpy(w).unsqueeze(0).to(DEV), wav_lens=torch.tensor([1.0]).to(DEV), normalize=False)
        e = F.normalize(e.squeeze().float(), p=2, dim=0).cpu()
    else:
        x = wl_fe(w, sampling_rate=16000, return_tensors="pt").to(DEV)
        e = F.normalize(wl(**x).embeddings.squeeze().float(), p=2, dim=0).cpu()
    _EMB[(p, kind)] = e
    return e
def cos(a, b, kind): return float(F.cosine_similarity(emb(a, kind), emb(b, kind), dim=0))

def join_spelled_letters(t):          # "N E U" -> "NEU" (reviewer fix), applied to reference and hypothesis
    return re.sub(r"\b(?:[A-Z] )+[A-Z]\b", lambda m: m.group(0).replace(" ", ""), str(t))
def normalize_transcript(text):        # identical to Untitled13 after the letter-joining step
    text = unicodedata.normalize("NFKC", join_spelled_letters(text)).lower()
    for a, b in [("’", "'"), ("–", " "), ("—", " "), ("-", " ")]: text = text.replace(a, b)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return re.sub(r"\s+", " ", text).strip()
@torch.inference_mode()
def transcribe(p, which):
    proc, mdl, dtype = ASR[which]
    mi = proc(load_audio_16k(p), sampling_rate=16000, return_tensors="pt", return_attention_mask=True)
    args = dict(input_features=mi.input_features.to(DEV, dtype), language="en", task="transcribe", max_new_tokens=128)
    if getattr(mi, "attention_mask", None) is not None: args["attention_mask"] = mi.attention_mask.to(DEV)
    return proc.batch_decode(mdl.generate(**args), skip_special_tokens=True)[0].strip()
@torch.inference_mode()
def mos(p): return float(utmos(torch.from_numpy(load_audio_16k(p)).unsqueeze(0).to(DEV), 16000).item())

def score(out, text, src, cond_ref, eval_clips):
    r = dict(sim_ecapa_cond=cos(out, cond_ref, "ecapa"), sim_wavlm_cond=cos(out, cond_ref, "wavlm"),
             sim_ecapa_eval=float(np.mean([cos(out, c, "ecapa") for c in eval_clips])),
             sim_wavlm_eval=float(np.mean([cos(out, c, "wavlm") for c in eval_clips])),
             leak_ecapa_source=cos(out, src, "ecapa"), utmos=mos(out), duration_out=librosa.get_duration(path=out))
    ref_n = normalize_transcript(text)
    for which in ("small", "large"):
        hyp = transcribe(out, which); hyp_n = normalize_transcript(hyp)
        r[f"hyp_{which}"] = hyp; r[f"wer_{which}"] = jiwer.wer(ref_n, hyp_n); r[f"cer_{which}"] = jiwer.cer(ref_n, hyp_n)
        r[f"ref_words"] = len(ref_n.split())
    for k in list(_EMB):                 # keep only cached reference/eval/source embeddings
        if k[0] == out: _EMB.pop(k)
    return r
print("✅ Scoring models ready")

DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _speechbrain_save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _speechbrain_load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _recover
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


hyperparams.yaml: 0.00B [00:00, ?B/s]

DEBUG:speechbrain.utils.fetching:Fetch: Local file found, creating symlink '/root/.cache/huggingface/hub/models--speechbrain--spkrec-ecapa-voxceleb/snapshots/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286/hyperparams.yaml' -> '/content/evaluation_models/ecapa_voxceleb/hyperparams.yaml'
INFO:speechbrain.utils.fetching:Fetch custom.py: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for _save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for _load
DEBUG:speechbrain.utils.checkpoints:Registered parameter transfer hook for _load
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint save hook for save
DEBUG:speechbrain.utils.checkpoints:Registered checkpoint load hook for load_if_possible
DEBUG:speechbrain.utils.parameter_transfer:Collecting files (or symlinks) for pretraining in /content/evaluation_models/ecapa_voxceleb.
INFO:speechbrain.utils.fetching:Fetch embedding_mod

embedding_model.ckpt:   0%|          | 0.00/83.3M [00:00<?, ?B/s]

DEBUG:speechbrain.utils.fetching:Fetch: Local file found, creating symlink '/root/.cache/huggingface/hub/models--speechbrain--spkrec-ecapa-voxceleb/snapshots/0f99f2d0ebe89ac095bcc5903c4dd8f72b367286/embedding_model.ckpt' -> '/content/evaluation_models/ecapa_voxceleb/embedding_model.ckpt'
DEBUG:speechbrain.utils.parameter_transfer:Set local path in self.paths["embedding_model"] = /content/evaluation_models/ecapa_voxceleb/embedding_model.ckpt
INFO:speechbrain.utils.fetching:Fetch mean_var_norm_emb.ckpt: Fetching from HuggingFace Hub 'speechbrain/spkrec-ecapa-voxceleb' if not cached


mean_var_norm_emb.ckpt:   0%|          | 0.00/1.92k [00:00<?, ?B/s]

... (26 outputs left out)

pytorch_model.bin:   0%|          | 0.00/405M [00:00<?, ?B/s]

Downloading: "https://github.com/tarepan/SpeechMOS/zipball/v1.2.0" to /root/.cache/torch/hub/v1.2.0.zip
Downloading: "https://github.com/tarepan/SpeechMOS/releases/download/v1.0.0/utmos22_strong_step7459_v1.pt" to /root/.cache/torch/hub/checkpoints/utmos22_strong_step7459_v1.pt


100%|██████████| 392M/392M [00:05<00:00, 73.3MB/s]


✅ Scoring models ready


In [ ]:
# Cell 6 — REPRODUCTION CHECK (must pass)
H40 = pd.read_csv(HOLDOUT40_META)
f = os.path.join(EXISTING, "seed_based_outputs", "optimized_hybrid", "optimized_hybrid_sample_01_tau_0.5_steps_10.wav")
s = score(f, H40.transcript[0], H40.prepared_wav[0], TARGET_REF, EVAL_CLIPS)
print(f"Existing output: similarity {s['sim_ecapa_cond']:.4f} (paper CSV 0.6885) | WER small {s['wer_small']:.4f} (paper 0.1667)")
assert abs(s["sim_ecapa_cond"] - 0.6885) < 0.01, "❌ scoring differs from the paper — stop and report"
new = os.path.join(OUT, "_check_selected_s01.wav")
run_openvoice(H40.prepared_wav[0], TARGET_REF, 0.5, new.replace(".wav", "_ov.wav"), 202701)
run_seedvc(new.replace(".wav", "_ov.wav"), TARGET_REF, 10, new, 202901)
t = score(new, H40.transcript[0], H40.prepared_wav[0], TARGET_REF, EVAL_CLIPS)
print(f"Regenerated with the paper's seeds: similarity {t['sim_ecapa_cond']:.4f} | transcript: {t['hyp_small']}")
print("✅ REPRODUCTION CHECK PASSED" if abs(t["sim_ecapa_cond"] - 0.6885) < 0.03 else "⚠️ regenerated output differs — report before continuing")

Existing output: similarity 0.6885 (paper CSV 0.6885) | WER small 0.1667 (paper 0.1667)


It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
It is strongly recommended to pass the `sampling_rate` argument to this function. Failing to do so can result in silent errors that might be hard to debug.
100%|██████████| 10/10 [00:00<00:00, 18.53it/s]


RTF: 0.23953276860449357
Regenerated with the paper's seeds: similarity 0.6873 | transcript: We expect that invading will be available to help you shortly.
✅ REPRODUCTION CHECK PASSED
